In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import time
import os
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from scipy.stats import ttest_rel
from scipy.stats import rankdata
from scipy.stats import shapiro
from scipy.stats import wilcoxon
plt.rcParams['font.family'] = 'Times New Roman'
pd.set_option('future.no_silent_downcasting', True)

## Constants

In [2]:
NUM_DECIMAL_PLACES = 4
VARIANTS = np.round(np.linspace(0.0, 1.0, 10001), NUM_DECIMAL_PLACES)
N_VARIANTS = VARIANTS.size
DELTA = VARIANTS[1] - VARIANTS[0]

EVALS = 1000
PLATEAU = 300

FIXED_K = 2
FIXED_D = 0.8

THRESHOLD = 0.05
SUBSET = False 
SUBSET_TYPE = 'NA' # Either 'a', 'b', or 'NA' 
if not SUBSET:
    SUBSET_TYPE = 'NA' 
RESULTS_FILENAME = 'your-filename-here-1.csv'

## Functions

In [3]:
%run helper_functions.ipynb

In [4]:
def get_pdf_normal(row, σ, μ, get_full_dist = False):  
    """
    Note: row is assumed to have a valid (non-NA) 'revised_guess'
    """
    y = norm.pdf(VARIANTS, μ, σ) # Since VARIANTS is in [0,1], area may be < 1
    y = y/y.sum() # Now, sum of y is 1
    y = y*N_VARIANTS

    if get_full_dist:
        return [VARIANTS, y]
    else: # Find the y value of the revised guess
        revised = float(row['revised_guess'])
        index_variant = int(round(revised / DELTA))
        return y[index_variant]

In [5]:
def get_pdf_conform(row, σ, α, d, k, get_full_dist = False):
    """
    Note: row is assumed to have a valid (non-NA) 'independent_guess' and 'revised_guess'
    """
    ind_guess = row['independent_guess'] 
    alters = ['alter_1_ind_guess', 'alter_2_ind_guess', 'alter_3_ind_guess']
    x_vec = [ind_guess] + [row[l] for l in alters if not np.isnan(row[l])]
    
    if len(x_vec) == 1: # Only the independent guess is included 
        μ = ind_guess
        return get_pdf_normal(row, σ, μ, get_full_dist)
    
    x_vec.sort() 
    ind_guess_index = x_vec.index(ind_guess)
    weight_each_guess = [1] * len(x_vec) 
    num_alter_guesses = len(x_vec)-1 
    weight_each_guess[ind_guess_index] = α
    
    if d != 0: 
        # Get probabilities without weighting the guesses 
        prob_social = prob_choose(x_vec, d, k) # Note x_vec is sorted
        # Apply weight_each_guess
        prob_each_guess = [weight_each_guess[i]*prob_social[i] for i in range(len(prob_social))]
        normalizer = sum(prob_each_guess)
        prob_each_guess = [i/normalizer for i in prob_each_guess]
    elif d == 0:
        normalizer = sum(weight_each_guess)
        prob_each_guess = [i/normalizer for i in weight_each_guess]
    
    y_total = np.zeros(N_VARIANTS)
    means = np.array(x_vec)
    probs = np.array(prob_each_guess)
    y_all = norm.pdf(VARIANTS[np.newaxis, :], means[:, np.newaxis], σ)
    y_all = y_all / y_all.sum(axis=1, keepdims=True)
    y_all = y_all * probs[:, np.newaxis]
    y_total = y_all.sum(axis=0)
        
    if abs(y_total.sum()-1) > 1e-8:
        raise ValueError("The sum of y_total should be 1 in this step.")
    y_total = y_total * N_VARIANTS

    if get_full_dist:
        return [VARIANTS, y_total]          
    else:
        revised = float(row['revised_guess'])
        index_variant = int(round(revised / DELTA))
        return y_total[index_variant]

In [6]:
def precompute_fdg2(data):
    """
    Note: data will not have any NAs for 'independent_guess' after first 
    line (OK if there are NAs for revised guess here). 
    """
    data = data[data['independent_guess'].notna()]
    grouped = data.groupby(['game_id', 'round_index'])
    precomputed = {}
    for (game_id, round_index), group_df in grouped:
        player_ids = group_df['player_id'].unique().tolist()
        matrix_size = len(player_ids)
        player_index_map = {player_id: idx for idx, player_id in enumerate(player_ids)}
    
        mapped_df = group_df.replace(player_index_map)
        mapped_df = mapped_df.sort_values(by='player_id', ascending=True)
        
        s = np.array(mapped_df['independent_guess'].tolist())
        if pd.Series(s).isna().any():
            raise ValueError("There are NA values in s vector")
        
        A = np.zeros((matrix_size, matrix_size))
        has_alters = np.zeros(matrix_size, dtype=bool)
        
        for _, row in mapped_df.iterrows():
            player_idx = row['player_id'] # A number
            for l in range(1, 4):
                alter_idx = row[f'alter_{l}'] 
                # If alter's independent guess was NA, they do not contribute
                if not pd.isna(row[f'alter_{l}_ind_guess']): 
                    A[player_idx, alter_idx] = 1
                    has_alters[player_idx] = True
            
        precomputed[(game_id, round_index)] = {'s': s, 
                                               'A': A, 
                                               'player_index_map': player_index_map,
                                               'has_alters': has_alters}
    return precomputed

In [7]:
def result_one_row(row, model_type, σ, α, d, k, precomputed, get_full_dist = False):
    if model_type == 'conf': 
        return get_pdf_conform(row, σ, α, d, k, get_full_dist)
        
    elif model_type == 'fdg1': 
        own = row['independent_guess']
        alter_cols = ['alter_1_ind_guess', 'alter_2_ind_guess', 'alter_3_ind_guess']
        alters = [row[c] for c in alter_cols if pd.notna(row[c])]
        x = np.array([own] + alters)
        w = np.ones(len(x))
        w[0] = float(α)
        w = w / w.sum()
        predicted_FDG1 = float(np.dot(w, x))
        new_σ = float(σ * np.sqrt(np.sum(w ** 2)))
        return get_pdf_normal(row, new_σ, predicted_FDG1, get_full_dist)

    else: # fdg2
        if pd.isna(row['independent_guess']) or pd.isna(row['revised_guess']):
            return np.nan
        key = (row['game_id'], row['round_index'])
        pre = precomputed[key]
        focal_player_idx = pre['player_index_map'][row['player_id']]
        s = pre['s']
        M = pre['A'].copy()
        for i in range(len(s)):
            if pre['has_alters'][i]: M[i, i] = α
            else: M[i, i] = 1
        row_sums = M.sum(axis=1, keepdims=True)
        M = M / row_sums         
        M_squared = M @ M
        all_predicted_FDG2 = M_squared @ s
        predicted_FDG2 = all_predicted_FDG2[focal_player_idx]

        w = M_squared[focal_player_idx, :]
        new_σ = σ * np.sqrt(np.sum(w ** 2))
        return get_pdf_normal(row, new_σ, predicted_FDG2, get_full_dist)

In [8]:
def run_model(data, model_type, σ, α, d, k, stat, subset, subset_type, precomputed, responsive_players):
    if model_type != 'fdg2': 
        # Do all data filtering and subsetting before getting results, since results are per-row
        clean_df = data[data['independent_guess'].notna() & data['revised_guess'].notna()].copy()
        if subset:
            if subset_type == 'a':
                mask = clean_df['independent_guess'] != clean_df['revised_guess']
            elif subset_type == 'b':
                mask = clean_df['player_id'].isin(responsive_players)
            clean_df = clean_df[mask] 
            player_ids = clean_df.loc[mask, 'player_id'].values
        else:
            player_ids = clean_df['player_id'].values
            
        results = clean_df.apply(lambda row: result_one_row(row, model_type, σ, α, d, k, precomputed),axis=1)
        
    else: # fdg2
        # Do not fully subset the data before getting results, since need more data to build matrix 
        new_df = data[data['independent_guess'].notna()].copy()
        results = new_df.apply(lambda row: result_one_row(row, model_type, σ, α, d, k, precomputed),axis=1)

        if subset:
            if subset_type == 'a':
                mask = new_df['independent_guess'] != new_df['revised_guess']
            elif subset_type == 'b':
                mask = new_df.loc[results.index, 'player_id'].isin(responsive_players).values
            results = results[mask]
    
        # Finally, remove NAs from results (when revised guess was NA but independent guess wasn't)
        # Now, this has equivalent data to the results from clean_df 
        results = results.dropna()

        # Get player_ids for fdg2 
        if stat == 'results': 
            clean_df = data[data['independent_guess'].notna() & data['revised_guess'].notna()].copy()        
            if subset:
                if subset_type == 'a':
                    mask = clean_df['independent_guess'] != clean_df['revised_guess']
                elif subset_type == 'b':
                    mask = clean_df['player_id'].isin(responsive_players)
                clean_df = clean_df[mask] 
                player_ids = clean_df.loc[mask, 'player_id'].values
            else:
                player_ids = clean_df['player_id'].values
            
    if stat == 'mean':
        return np.mean(results)
    elif stat == 'median':
        return np.median(results)
    elif stat == 'sumlogs':
        if np.all(results > 0):
            return np.sum(np.log(results))
        else:
            return -1e100 # A large negative number
    elif stat == 'results':
        return [results, player_ids]

In [9]:
def get_data(selected_condition):
    full_data = pd.read_csv('all_studies_round_data.csv')
    num_rounds = len(full_data['round_index'].unique())
    if num_rounds != 20:
        raise ValueError("Number of rounds is not 20")
        
    selected_data = full_data[full_data['condition'] == selected_condition]
    game_id_list = selected_data['game_id'].unique().tolist()

    if selected_data['independent_guess'].min() < 0:
        raise ValueError("Minimum of independent_guess is < 0")
    if selected_data['revised_guess'].min() < 0:
        raise ValueError("Minimum of revised_guess is < 0")

    selected_data = selected_data.assign(alter_1_ind_guess = np.nan, alter_2_ind_guess = np.nan,
                                     alter_3_ind_guess = np.nan, alter_1_rev_guess = np.nan,
                                     alter_2_rev_guess = np.nan, alter_3_rev_guess = np.nan)
    data = pd.DataFrame(columns=selected_data.columns)
    data_chunks = []

    for id_of_game in game_id_list:
        df = selected_data[selected_data['game_id'] == id_of_game] 
    
        for round_index in range(1, num_rounds+1):
            round_df = df[df['round_index'] == round_index].copy()
            ind_guess_map = round_df.set_index('player_id')['independent_guess'].to_dict()
            rev_guess_map = round_df.set_index('player_id')['revised_guess'].to_dict() 
    
            for idx, row in round_df.iterrows():
                for i, alter in enumerate(['alter_1', 'alter_2', 'alter_3'], 1):
                    alter_id = row[alter]
                    ind_guess = ind_guess_map.get(alter_id)
                    rev_guess = rev_guess_map.get(alter_id) 
    
                    round_df.at[idx, f'alter_{i}_ind_guess'] = ind_guess
                    round_df.at[idx, f'alter_{i}_rev_guess'] = rev_guess
                    
            data_chunks.append(round_df)
    
    final_data = pd.concat(data_chunks, ignore_index=True) 
    
    return final_data.copy()

In [10]:
def make_objective(data, model_type, infer_d, infer_k, stat, subset, subset_type, precomputed, responsive_players, fixed_d = FIXED_D, fixed_k = FIXED_K):
    
    def objective(params):
        sigma = float(params["sigma"])
        alpha = float(params["alpha"])
        d = float(params["d"]) if infer_d else fixed_d
        k = int(params["k"]) if infer_k else fixed_k
        
        score = run_model(data, model_type, sigma, alpha, d, k, stat, subset, subset_type, precomputed, responsive_players)
        
        return {"loss": -float(score), "status": STATUS_OK}

    return objective

In [11]:
def make_early_stop(plateau):
    def early_stop_fn(trials, *args):
        if len(trials.trials) <= plateau:
            return False, {}
        losses = [t["result"]["loss"] for t in trials.trials]
        best_overall = min(losses)
        best_before_window = min(losses[:-plateau])
        return best_overall >= best_before_window, {}
    return early_stop_fn

In [12]:
def save_results(model_type, subset, subset_type, stat, selected_condition, best, first_idx, max_score, score, filename):
    row = {"model": model_type,
           "subset": subset,
           "subset_type": subset_type,
           "statistic": stat, 
           "selected_condition": selected_condition,
           "sigma": best["sigma"], 
           "alpha": best["alpha"], 
           "d": best.get("d", FIXED_D),
           "k": best.get("k", FIXED_K),
           "first_idx": first_idx,
           "max_score": max_score, 
           "score": score}
    df = pd.DataFrame([row])
    write_header = not os.path.exists(filename)
    df.to_csv(filename, mode="a", index=False, header=write_header)

In [13]:
def classify_players(data, threshold):
    clean_df = data[data['independent_guess'].notna() & data['revised_guess'].notna()].copy()
    clean_df['abs_revision'] = (clean_df['independent_guess'] - clean_df['revised_guess']).abs()
    
    player_stats = clean_df.groupby('player_id')['abs_revision'].agg(['mean', 'max', 'median'])
    anchored = set(player_stats[player_stats['median'] < threshold].index)
    responsive = set(player_stats[player_stats['median'] >= threshold].index)
    
    return anchored, responsive

In [14]:
def compare_and_test(x, y):
    diff = x.reset_index(drop=True) - y.reset_index(drop=True)
    if np.all(diff == 0):
        return {'shapiro_stat': np.nan, 
                'shapiro_p': np.nan, 
                'wilcoxon_stat': np.nan, 
                'wilcoxon_p': np.nan,
                'diff_mean': np.nan, 
                'diff_median': np.nan,
                'W_diff': np.nan, 
                'W_stat': np.nan}

    shapiro_stat, shapiro_p = shapiro(diff)
    wilcoxon_stat, wilcoxon_p = wilcoxon(x, y)

    signed_diff = diff[diff != 0] 
    abs_diff = np.abs(signed_diff)
    ranks = rankdata(abs_diff, method="average")
    signed_ranks = np.sign(signed_diff) * ranks
    W_plus = np.sum(signed_ranks[signed_ranks > 0])
    W_minus = -np.sum(signed_ranks[signed_ranks < 0]) 

    return {'shapiro_stat': shapiro_stat,
            'shapiro_p': shapiro_p, 
            'wilcoxon_stat': wilcoxon_stat,
            'wilcoxon_p': wilcoxon_p,
            'diff_mean': diff.mean(),
            'diff_median': diff.median(),
            'W_diff': W_plus - W_minus,
            'W_stat': np.sum(signed_ranks)}

## Analysis

### Load the data

Data obtained from [here](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/EOYZKH).

In [15]:
full_data = pd.read_csv('all_studies_round_data.csv')
num_rounds = len(full_data['round_index'].unique())
print(num_rounds)

20


In [16]:
full_data['condition'].unique()

array(['dynamic', 'static', 'dynamic_no_feedback',
       'dynamic_full_feedback', 'dynamic_self_feedback',
       'solo_no_feedback', 'solo_feedback'], dtype=object)

### Run the models to find best-fitting parameters

In [17]:
RESULTS_FILENAME, THRESHOLD, SUBSET, SUBSET_TYPE

('your-filename-here-1.csv', 0.05, False, 'NA')

In [ ]:
start = time.time() 
model_type = 'conf'
infer_d = True
infer_k = True

for selected_condition in ['static', 'dynamic', 'dynamic_no_feedback',
                           'dynamic_self_feedback', 'dynamic_full_feedback']:
    data = get_data(selected_condition)  
    if SUBSET and SUBSET_TYPE == 'b':
        anchored, responsive = classify_players(data, THRESHOLD)
    else:
        anchored, responsive = None, None 

    train_data = data[data['round_index'] % 2 == 1]
    test_data = data[data['round_index'] % 2 == 0]

    if model_type != 'fdg2':
        precomputed_train = None 
        precomputed_test = None 
    else: 
        precomputed_train = precompute_fdg2(train_data)
        precomputed_test = precompute_fdg2(test_data)

    for stat in ['mean', 'median', 'sumlogs']:
        space = {"sigma": hp.uniform("sigma", 1e-4, 1.0),
                 "alpha": hp.loguniform("alpha", np.log(1e-4), np.log(1000.0))}
        if infer_d:
            space["d"] = hp.uniform("d", -0.99, 0.99)
        if infer_k:
            space["k"] = hp.quniform("k", 1, 3, 1)
        
        objective_fn = make_objective(train_data, model_type, infer_d, infer_k, stat, SUBSET, SUBSET_TYPE, precomputed_train, responsive)
        trials = Trials()
        best = fmin(fn=objective_fn, space=space, algo=tpe.suggest, max_evals=EVALS, 
                    trials=trials, rstate=np.random.default_rng(42),
                   early_stop_fn=make_early_stop(PLATEAU)) 
        
        scores = np.array([-t["result"]["loss"] for t in trials.trials])
        running_best = np.maximum.accumulate(scores)
        max_score = running_best.max() 
        first_idx = np.argmax(running_best == max_score) 

        best_k = best['k'] if infer_k else FIXED_K
        best_d = best['d'] if infer_d else FIXED_D
        best_α = best['alpha']
        best_σ = best['sigma']
        test_data_score = run_model(test_data, model_type, best_σ, best_α, best_d, best_k, stat, SUBSET, SUBSET_TYPE, precomputed_test, responsive)

        save_results(model_type, SUBSET, SUBSET_TYPE, stat, selected_condition, best, first_idx, max_score, test_data_score, RESULTS_FILENAME)

        print(f"{selected_condition}, {stat}")

end = time.time()
print("Time:", (end - start) / 60)

### Compare the models

In [18]:
THRESHOLD, SUBSET, SUBSET_TYPE

(0.05, False, 'NA')

In [ ]:
FDG1_FILENAME = 'your-filename-here-2.csv'
FDG2_FILENAME = 'your-filename-here-3.csv'
CONF_FILENAME = 'your-filename-here-4.csv'

df_fdg1 = pd.read_csv(FDG1_FILENAME)
df_fdg2 = pd.read_csv(FDG2_FILENAME)
df_conf = pd.read_csv(CONF_FILENAME)

In [ ]:
results_fdg1_conf = []
results_fdg2_conf = []
results_fdg1_fdg2 = []

results_null_conf = []
results_null_fdg1 = []
results_null_fdg2 = []

for selected_condition in ['static', 'dynamic', 'dynamic_no_feedback',
                           'dynamic_self_feedback', 'dynamic_full_feedback']:
    selected_data = full_data[full_data['condition'] == selected_condition]
    data = get_data(selected_condition)     
    test_data = data[data['round_index'] % 2 == 0]
    precomputed_test = precompute_fdg2(test_data)

    anchored, responsive = classify_players(data, THRESHOLD)

    for stat in ['mean', 'median', 'sumlogs']: 
        row_conf = df_conf[(df_conf['subset'] == SUBSET) & (df_conf['selected_condition'] == selected_condition) & (df_conf['statistic'] == stat)]
        alpha_conf = row_conf['alpha'].values[0]
        sigma_conf = row_conf['sigma'].values[0]
        d_conf = row_conf['d'].values[0]
        k_conf = row_conf['k'].values[0]
        results_conf, player_ids = run_model(test_data, 'conf', sigma_conf, alpha_conf, 
                                 d_conf, k_conf, 'results', SUBSET, SUBSET_TYPE, None, responsive)

        row_fdg1 = df_fdg1[(df_fdg1['subset'] == SUBSET) & (df_fdg1['selected_condition'] == selected_condition) & (df_fdg1['statistic'] == stat)]
        alpha_fdg1 = row_fdg1['alpha'].values[0]
        sigma_fdg1 = row_fdg1['sigma'].values[0]
        results_fdg1, player_ids = run_model(test_data, 'fdg1', sigma_fdg1, alpha_fdg1, 
                                 FIXED_D, FIXED_K, 'results', SUBSET, SUBSET_TYPE, None, responsive)

        row_fdg2 = df_fdg2[(df_fdg2['subset'] == SUBSET) & (df_fdg2['selected_condition'] == selected_condition) & (df_fdg2['statistic'] == stat)]
        alpha_fdg2 = row_fdg2['alpha'].values[0]
        sigma_fdg2 = row_fdg2['sigma'].values[0]
        results_fdg2, player_ids = run_model(test_data, 'fdg2', sigma_fdg2, alpha_fdg2, 
                                 FIXED_D, FIXED_K, 'results', SUBSET, SUBSET_TYPE, precomputed_test, responsive)

        clean_df = test_data[test_data['independent_guess'].notna() & test_data['revised_guess'].notna()].copy()

        # Average y* per individual to address within-person dependence
        avg_conf = results_conf.groupby(player_ids).mean()
        avg_fdg1 = results_fdg1.groupby(player_ids).mean()
        avg_fdg2 = results_fdg2.groupby(player_ids).mean()

        t_stat_1, p_val_1 = ttest_rel(avg_fdg1.values, avg_conf.values)
        w_fdg1_conf = compare_and_test(avg_fdg1, avg_conf)
        results_fdg1_conf.append({
            'subset': SUBSET,
            'subset_type': SUBSET_TYPE,
            'condition': selected_condition,
            'statistic': stat,
            't_stat': t_stat_1,
            'p_val': p_val_1,
            'alpha_conf': alpha_conf,
            'sigma_conf': sigma_conf,
            'alpha_fdg1': alpha_fdg1,
            'sigma_fdg1': sigma_fdg1,
            **w_fdg1_conf
        })

        t_stat_2, p_val_2 = ttest_rel(avg_fdg2.values, avg_conf.values)
        w_fdg2_conf = compare_and_test(avg_fdg2, avg_conf)
        results_fdg2_conf.append({
            'subset': SUBSET,
            'subset_type': SUBSET_TYPE,
            'condition': selected_condition,
            'statistic': stat,
            't_stat': t_stat_2,
            'p_val': p_val_2,
            'alpha_conf': alpha_conf,
            'sigma_conf': sigma_conf,
            'alpha_fdg2': alpha_fdg2,
            'sigma_fdg2': sigma_fdg2,
            **w_fdg2_conf
        })

        t_stat_3, p_val_3 = ttest_rel(avg_fdg1.values, avg_fdg2.values)
        w_fdg1_fdg2 = compare_and_test(avg_fdg1, avg_fdg2)
        results_fdg1_fdg2.append({
            'subset': SUBSET,
            'subset_type': SUBSET_TYPE,
            'condition': selected_condition,
            'statistic': stat,
            't_stat': t_stat_3,
            'p_val': p_val_3,
            'alpha_fdg1': alpha_fdg1,
            'sigma_fdg1': sigma_fdg1,
            'alpha_fdg2': alpha_fdg2,
            'sigma_fdg2': sigma_fdg2,
            **w_fdg1_fdg2
        })

        ones = [1]*len(avg_conf.values)
        
        t_stat_4, p_val_4 = ttest_rel(ones, avg_conf.values)
        w_ones_conf = compare_and_test(pd.Series(ones), avg_conf)
        results_null_conf.append({
            'subset': SUBSET,
            'subset_type': SUBSET_TYPE,
            'condition': selected_condition,
            'statistic': stat,
            't_stat': t_stat_4,
            'p_val': p_val_4,
            'alpha_conf': alpha_conf,
            'sigma_conf': sigma_conf,
            **w_ones_conf
        })

        t_stat_5, p_val_5 = ttest_rel(ones, avg_fdg1.values)
        w_ones_fdg1 = compare_and_test(pd.Series(ones), avg_fdg1)
        results_null_fdg1.append({
            'subset': SUBSET,
            'subset_type': SUBSET_TYPE,
            'condition': selected_condition,
            'statistic': stat,
            't_stat': t_stat_5,
            'p_val': p_val_5,
            'alpha_fdg1': alpha_fdg1,
            'sigma_fdg1': sigma_fdg1,
            **w_ones_fdg1
        })

        t_stat_6, p_val_6 = ttest_rel(ones, avg_fdg2.values)
        w_ones_fdg2 = compare_and_test(pd.Series(ones), avg_fdg2)
        results_null_fdg2.append({
            'subset': SUBSET,
            'subset_type': SUBSET_TYPE,
            'condition': selected_condition,
            'statistic': stat,
            't_stat': t_stat_6,
            'p_val': p_val_6,
            'alpha_fdg2': alpha_fdg2,
            'sigma_fdg2': sigma_fdg2,
            **w_ones_fdg2
        })
        
df_fdg1_conf = pd.DataFrame(results_fdg1_conf)
df_fdg2_conf = pd.DataFrame(results_fdg2_conf)
df_fdg1_fdg2 = pd.DataFrame(results_fdg1_fdg2)

df_null_conf = pd.DataFrame(results_null_conf)
df_null_fdg1 = pd.DataFrame(results_null_fdg1)
df_null_fdg2 = pd.DataFrame(results_null_fdg2)